Zellen haben eine komplexe Binnenstruktur. In dieser Aufgabe betrachten wir schematisch einen Transportprozess innerhalb einer idealisierten Zelle. Dabei soll eine Substanz per Diffusion vom Zellkern (blau) zur Zellmembran (rot) transportiert werden. Es gibt hierfür einen durchgehenden Weg (hellgrau), allerdings gibt es Bereiche (dunkelgrau), in denen die Diffusion, z.B. durch bestehende Strukturen, stark verändert wird.

![test](simple-cell.png)

Wir betrachten die Konzentration $u$ einer Substanz und nehmen an, dass sich die Verteilung durch die Diffusionsgleichung
$$ \frac{\partial u}{\partial t} = \nabla \cdot \left( -\alpha \nabla u \right) = 0 $$
beschreiben lässt. Die Diffusionskonstante $\alpha$ ist dabei ortsabhängig. Sie beträgt $\alpha = 1 mm^2/s$ im (hellgrauen) freien Bereich sowie $\alpha = \epsilon = 10^{-3} mm^2/s$ im (dunkelgrauen) Bereich mit Barriere.  Die Randbedingungen sind so, dass am Zellkern wird die Substanz mit der Konzentration $1\;mmol$ angeliefert wird. An der Zellmembran  wird die Substanz vollständig abgeführt wird, d.h., die Konzentration dort betrage $0\;mmol$.
Ermitteln Sie numerisch den Konzentrationsverlauf im Gleichgewicht. Dazu können Sie Ihre Lösung z.B. als vtu Datei exportieren und mittels Paraview visualisieren.


Als erstes importieren wir die Python Distribution von UG4:

In [3]:
import ug4py.pyugcore as ug4
import ug4py.pylimex as limex
import ug4py.pyconvectiondiffusion as cd
import sys

from scipy.special.cython_special import mathieu_modsem1

# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, '../')
import modsimtools as util

Nun erstellen wir das Simulationsgebiet (Domain)

In [5]:
requiredSubsets = ["FREE", "NUCLEUS", "MEMBRANE", "BARRIER"]
gridName = "simple-cell.ugx"
numRefs = 2
dom = util.CreateDomain(gridName, numRefs, requiredSubsets)

Loading Domain 'simple-cell.ugx'...
Domain loaded.
Refining ...
Refining step {0} ...
Refining step {1} ...
Refining done


Auf dem Gebiet definieren wir einen Ansatzraum. Hier werden unsere Unbekannten, welche auf den Gitterknoten definiert sind, durch Interpolationsfunktionen auf das gesamte Simulationsgebiet erweitert (diskrete Werte an Knoten -> Feld auf dem Gebiet).

In [9]:
approxSpace = ug4.ApproximationSpace2d(dom)
approxSpace.add_fct("u", "Lagrange", 1)
# enable multigrid
approxSpace.init_levels()
approxSpace.init_surfaces()
approxSpace.init_top_surface()
approxSpace.print_statistic()

| ----------------------------------------------------------------------------------------- |
|  Number of DoFs (All Procs)                                                               |
|  Algebra: Block 1 (divide by 1 for #Index)                                                |
|                                                                                           |
|    GridLevel   |       Domain |      0: FREE |   1: NUCLEUS |  2: MEMBRANE |   3: BARRIER |
| ----------------------------------------------------------------------------------------- |
| (lev,    0)    |          216 |            9 |            6 |           48 |          153 |
| (lev,    1)    |          810 |          252 |           12 |           96 |          450 |
| (lev,    2)    |         3132 |         1431 |           24 |          192 |         1485 |
| (lev,    0, g) |          216 |            9 |            6 |           48 |          153 |
| (lev,    1, g) |          810 |          252 |           1

Nun kümmern wir uns um die Beschreibung unserer Differentialgleichung. Dafür Diskretisieren wir elementweise unsere PDE. Im Modul **ConvectionDiffusion** steht dafür eine Art Baukasten zur Verfügung. Im allgemeinen können hier PDE's der Form

$$\partial_t (m_1 c + m_2) - \nabla \cdot \left ( D \nabla c - \vec{v} c - \vec{F} \right ) + r_1 \cdot c + r_2 = f + \nabla \cdot \vec{f}_2$$

implementiert werden. Jedem Term in obiger Gleichung kann ein Wert oder eine Funktion zugewiesen werden. Betrachten wir unsere PDE
$$ \frac{\partial u}{\partial t} = \nabla \cdot \left( -\alpha \nabla u \right) = 0 $$
müssen wir also $m_1 = 1$ und $D = \alpha$ setzen (Rest ist 0!).


In [27]:
alpha_barrier   = 0.00001
alpha_free      = 1.0
m1              = 1

# create element discretizations for each subdomain
elemDisc = {}
elemDisc["BARRIER"] = cd.ConvectionDiffusionFV12d("u", "BARRIER")
elemDisc["FREE"]    = cd.ConvectionDiffusionFV12d("u", "FREE")

# set mass scale
elemDisc["BARRIER"].set_mass_scale(m1)
elemDisc["FREE"].set_mass_scale(m1)

# set diffusion coefficients
elemDisc["BARRIER"].set_diffusion(alpha_barrier)
elemDisc["FREE"].set_diffusion(alpha_free)


Betrachten wir nun unsere Randbedingungen. In der Aufgabenstellung sind feste Werte für die Konzentration unserer Unbekannten gegeben. Deshalb müssen wir für unser Problem Dirichlet Randbedingungen am Nukleus und an der Membran definieren.

In [28]:
dirichletBND = ug4.DirichletBoundary2dCPU1()
dirichletBND.add(1.0, "u", "NUCLEUS")
dirichletBND.add(0.0, "u", "MEMBRANE")

Nun haben wir unser Problem vollständig definiert. Wir haben Diskretisierungen der partiellen Differentialgleichung auf den Elementen und Randbedingungen am Rand. in UG4 müssen wir diese zu einer Gebietsdiskretisierung zusammenfassen, welche das vollständige Problem beschreibt..

In [29]:
domainDisc = ug4.DomainDiscretization2dCPU1(approxSpace)
domainDisc.add(elemDisc["FREE"])
domainDisc.add(elemDisc["BARRIER"])
domainDisc.add(dirichletBND)

Wir werden das zeitabhängige Problem mit einem impliziten Eulerverfahren lösen. Unsere Unbekannten werden in einer sogenannten **Gitterfunktion** gespeichert. Über diese interpolieren wir einen Anfangswert und erhalten so ein vollständig definiertes Anfangswertproblem

In [30]:
# Create Solver and Time Discretization
lsolver=ug4.LUCPU1()
startTime = 0.0
endTime = 500.0
dt = 1
timeDisc=ug4.ThetaTimeStepCPU1(domainDisc, 1.0)
timeInt = limex.ConstStepLinearTimeIntegrator2dCPU1(timeDisc)
timeInt.set_linear_solver(lsolver)
timeInt.set_time_step(dt)
# Create GridFunction
usol = ug4.GridFunction2dCPU1(approxSpace)
# interpolate initial value
ug4.Interpolate(0.0, usol, "u")

Starten des Lösungsprozesses:

In [31]:
def MyVTKCallback(usol, step, time, dt) :
    ug4.WriteGridFunctionToVTK(usol, "vtk/SkinDiffusion_"+str(int(step)).zfill(5)+".vtu")

vtkobserver = ug4.PythonCallbackObserver2dCPU1(MyVTKCallback)
timeInt.attach_observer(vtkobserver)
try:
    timeInt.apply(usol, endTime, usol, startTime)
except Exception as inst:
    print(inst)

+++ Integrating: [	0	, 	500	] with dt=	1(500 iters)
+++ Const timestep +++1(t=0, dt=1)
+++ Assemble (t=0, dt=1)

ILUT: please use 'set_ordering_algorithm(..)' in the future
+++ Const timestep +++2(t=1, dt=1)
+++ Const timestep +++3(t=2, dt=1)
+++ Const timestep +++4(t=3, dt=1)
+++ Const timestep +++5(t=4, dt=1)
+++ Const timestep +++6(t=5, dt=1)
+++ Const timestep +++7(t=6, dt=1)
+++ Const timestep +++8(t=7, dt=1)
+++ Const timestep +++9(t=8, dt=1)
+++ Const timestep +++10(t=9, dt=1)
+++ Const timestep +++11(t=10, dt=1)
+++ Const timestep +++12(t=11, dt=1)
+++ Const timestep +++13(t=12, dt=1)
+++ Const timestep +++14(t=13, dt=1)
+++ Const timestep +++15(t=14, dt=1)
+++ Const timestep +++16(t=15, dt=1)
+++ Const timestep +++17(t=16, dt=1)
+++ Const timestep +++18(t=17, dt=1)
+++ Const timestep +++19(t=18, dt=1)
+++ Const timestep +++20(t=19, dt=1)
+++ Const timestep +++21(t=20, dt=1)
+++ Const timestep +++22(t=21, dt=1)
+++ Const timestep +++23(t=22, dt=1)
+++ Const timestep +++24(t=23,

Wiederholen Sie die Experimente für eine homogene Einbettung ($\epsilon = 1 mm^2/s$) sowie für eine Einbettung, in welcher der Transport sehr schnell abläuft ($\epsilon = 10^{3} mm^2/s$). Was ändert sich?

In [20]:
ug4.WriteGridFunctionToVTK(usol, "vtk/cell_final.vtu")